# Group Representations II — Companion Notebook

**6.7970/8.750 Symmetry and its Application to Machine Learning**

This notebook follows the Group Representations II exercise section by section. Use it to **prototype your code** and **test your implementations** against the course library before submitting on the website.

Each section includes small tests you can use to check your work.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atomicarchitects/symm4ml-colabs/blob/main/rep2_companion.ipynb)

## Setup

In [1]:
%%capture
!pip install https://symm4ml.mit.edu/_static/symm4ml_s26/symm4ml/symm4ml_latest.zip

In [2]:
import itertools
from typing import List, Set, FrozenSet
from pprint import pprint
import random

import numpy as np

from symm4ml import groups, linalg, rep

### Reference data

These matrices and tables are used throughout the exercise for testing.

In [3]:
# P(3) multiplication table
p3_table = np.array([
    [0, 1, 2, 3, 4, 5],
    [1, 0, 4, 5, 2, 3],
    [2, 5, 0, 4, 3, 1],
    [3, 4, 5, 0, 1, 2],
    [4, 3, 1, 2, 5, 0],
    [5, 2, 3, 1, 0, 4],
])

# P(3) irreps for testing
p3_irrep_trivial = np.array([[[1.0]], [[1.0]], [[1.0]], [[1.0]], [[1.0]], [[1.0]]])

p3_irrep_sign = np.array([[[1.0]], [[-1.0]], [[-1.0]], [[-1.0]], [[1.0]], [[1.0]]])

p3_irrep_rot = np.array([
    [[1.0, 0.0], [0.0, 1.0]],
    [[-0.2916587, 0.95652245], [0.95652245, 0.2916587]],
    [[-0.6825434, -0.73084507], [-0.73084507, 0.6825434]],
    [[0.97420209, -0.22567739], [-0.22567739, -0.97420209]],
    [[-0.5, 0.8660254], [-0.8660254, -0.5]],
    [[-0.5, -0.8660254], [0.8660254, -0.5]],
])

# D2 (Klein four-group) table
ans_table2 = np.array([[0, 1, 2, 3], [1, 0, 3, 2], [2, 3, 0, 1], [3, 2, 1, 0]])

# Z2 table
z2_table = np.array([[0, 1], [1, 0]])

---
## 1. `similarity_transform(rep, U)`

Compute the similarity transform $\rho'(g) = U \rho(g) U^{-1}$ of a representation. $U$ is an invertible (not necessarily unitary) complex matrix.

In [4]:
def similarity_transform(rep: np.array, U: np.array) -> np.array:
    """Returns transformed representation U rep U^{-1}.
    Input:
        rep: np.array [n, d, d] representation of the group. rep[i] is a matrix that
            represents the representation at the i-th element of group.
        U: np.array [d, d] invertible complex matrix
    Output:
        Transformed representation. np.array [n, d, d]
    """
    rep_prime = []
    for r in rep:
        right_half = np.inner(r, U)
        left_half = np.dot(U, right_half)
        rep_prime.append(left_half)
    return rep_prime

In [5]:
my_ans = similarity_transform(p3_irrep_rot, np.array([[0.0, 1.0], [1.0, 0.0]]))
print(my_ans)

class_ans = rep.similarity_transform(p3_irrep_rot, np.array([[0.0, 1.0], [1.0, 0.0]]))
print(class_ans)

[array([[1., 0.],
       [0., 1.]]), array([[ 0.2916587 ,  0.95652245],
       [ 0.95652245, -0.2916587 ]]), array([[ 0.6825434 , -0.73084507],
       [-0.73084507, -0.6825434 ]]), array([[-0.97420209, -0.22567739],
       [-0.22567739,  0.97420209]]), array([[-0.5      , -0.8660254],
       [ 0.8660254, -0.5      ]]), array([[-0.5      ,  0.8660254],
       [-0.8660254, -0.5      ]])]
[[[ 1.          0.        ]
  [ 0.          1.        ]]

 [[ 0.2916587   0.95652245]
  [ 0.95652245 -0.2916587 ]]

 [[ 0.6825434  -0.73084507]
  [-0.73084507 -0.6825434 ]]

 [[-0.97420209 -0.22567739]
  [-0.22567739  0.97420209]]

 [[-0.5        -0.8660254 ]
  [ 0.8660254  -0.5       ]]

 [[-0.5         0.8660254 ]
  [-0.8660254  -0.5       ]]]


---
## 2. `character_table(irreps, conj_classes)`

Compute the character table for a group given its irreps and conjugacy classes. The character of an irrep for a given class is the trace of its matrix representation for any element in that class.

the Character of the matrix representation $D^{\Gamma_j}(R)$ is

$\Chi^{\Gamma_j}(R) = tr D^{\Gamma_j}(R) = \sum_{\mu=1}^{\ell_j}D_{\mu \mu}^{\Gamma_j}(R)$

The character for each element in a conjugacy class is the same

Character Table `c_t`:
- Rows = irreps, columns = conjugacy classes

`c_t[i,j] = irrep[i]()`

In [ ]:
def character_table(
        irreps: List[np.array], 
        conj_classes: Set[FrozenSet[int]]
) -> np.ndarray:
    """Returns character table for a group.
    Input:
        irreps: List of np.arrays of shape [n, d, d], where n is the order of the group and d is the dimension of the irrep (which may vary).
        conj_classes: List of sets of integers, where the total number of integers across all sets in the list is n.
        Each set contains elements of a conjugacy class.
    Output:
        Character table. np.array [len(irreps), len(conj_classes)] (where the class / irrep order is unchanged)
    """

    c_table = np.zeros((len(irreps), len(conj_classes)))

    for i in range(0, len(irreps)):
        irrep = irreps[i]
        for j in range(0, len(conj_classes)):
            conj_class = list(list(conj_classes)[j])
            c_table[i,j] = np.trace(irrep[conj_class[0]])
    
    return c_table

In [13]:
p2_irreps = [[[[ 1.]],[[-1.]]], [[[1.]],[[1.]]]]

p2_conj_classes = {frozenset({1}), frozenset({0})}

p2_char_tab = character_table(p2_irreps, p2_conj_classes)

np.allclose(p2_char_tab, [[-1.0000000000000002, 1.0000000000000002], [1.0000000000000002, 1.0000000000000002]])

p3_irreps = [
    [[[1.]], [[1.]], [[1.]], [[1.]], [[1.]], [[1.]]],
    [[[ 1.+0.j]], [[-1.+0.j]], [[-1.+0.j]], [[-1.+0.j]], [[ 1.+0.j]], [[ 1.+0.j]]]
]
p3_conj_classes = {frozenset({1, 2, 3}), frozenset({4, 5}), frozenset({0})}

p3_char_tab = character_table(p3_irreps, p3_conj_classes)
np.allclose(p3_char_tab, [[1.0, 1.0, 1.0], [-1.0000000000000002, 1.0, 1.0], [2.220446049250313e-16, -1.0, 2.0]])

/tmp/ipython-input-2282455827.py:21: ComplexWarning: Casting complex values to real discards the imaginary part
  c_table[i,j] = np.trace(irrep[conj_class[0]])


ValueError: operands could not be broadcast together with shapes (6,3) (3,3) 

---
## 3. `regular_representation(table)`

Compute the left regular representation from a group's multiplication table. For a group with $h$ elements, $D(g)|h\rangle = |gh\rangle$.

See the [Wikipedia page](https://en.wikipedia.org/wiki/Regular_representation) for reference.

In [ ]:
def regular_representation(table: np.array) -> np.array:
    """Returns regular representation for group represented by a multiplication table.
    Input:
        table: np.array [n, n] where table[i, j] = k means i * j = k.
    Output:
        Regular representation. array [n, n, n] where reg_rep[i, :, :] = D(i) and D(i)e_j = e_{ij}.
        Equivalently, D(g) |h> = |gh>
    """
    n = len(table)
    reg_rep = []
    for i in range(0, n):
        right_side = np.zeros((n,n))
        for j in range(0,n):
            right_side[table[i,j],j] = 1
        reg_rep.append(np.inner(right_side, np.identity(n)))
    return reg_rep

In [7]:
pprint(np.identity(4).T.conj())

array([[1., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 1.]])


In [30]:
pprint(regular_representation(np.array([[0, 1], [1, 0]])))

assert np.allclose(
    regular_representation(np.array([[0, 1], [1, 0]])),
    np.array([[[1.0, 0.0], [0.0, 1.0]], [[0.0, 1.0], [1.0, 0.0]]]),
)

[array([[1., 0.],
       [0., 1.]]), array([[0., 1.],
       [1., 0.]])]


: 

In [29]:
my_ans = regular_representation(p3_table)
course_ans = rep.regular_representation(p3_table)

missing_in_mine = np.setdiff1d(course_ans, my_ans)
missing_in_course = np.setdiff1d(my_ans, course_ans)
print(missing_in_course)

[]


In [31]:
def unique_with_tol(a: np.array, *, tol: float):
    """Find unique elements of an array with a tolerance.
    Input:
        a: np.array of shape num_elements x d1 x ... x dm of which to find the unique elements
        tol: tolerance
    Output:
        centers: np.array of shape num_clusters x d1 x ... x dm containing the centers of the clusters
        inverses: np.array of shape num_elements containing the index of the corresponding center for each element of a
    Raises:
        ValueError: if the cluster are not clearly distinct
    Note:
        this function is "stable", the first element always belongs to the
        first cluster, the second element not in the first cluster belongs to the
        second cluster, etc.
    """
    assert a.ndim >= 1
    shape = a.shape
    a = a.reshape(len(a), -1)

    distances = np.linalg.norm(a[:, None] - a[None, :], axis=-1)
    inverses = -1 * np.ones(len(a), dtype=int)
    index = 0

    while True:
        (m,) = np.nonzero(inverses == -1)
        if len(m) == 0:
            break
        i = m[0]

        if np.any(inverses[distances[i] < tol] != -1):
            raise ValueError("The clusters are not clearly distinct.")

        inverses[distances[i] < tol] = index

        index += 1

    centers = np.zeros((np.max(inverses) + 1, a.shape[1]), dtype=a.dtype)
    np.add.at(centers, inverses, a)
    centers /= np.bincount(inverses)[:, None]

    centers = centers.reshape(len(centers), *shape[1:])
    return centers, inverses

def eigenspaces(
    val: np.ndarray, vec: np.ndarray, *, tol: float = 1e-8
) -> List[tuple[float, np.ndarray]]:
    """Regroup eigenvectors by eigenvalues.
    Input:
        val: eigenvalues (output of np.linalg.eig)
        vec: eigenvectors (output of np.linalg.eig)
        tol: tolerance for the eigenvalues similarity
    Output:
        list of (eigenvalue, eigenvectors) tuples
    """
    unique_val, i = unique_with_tol(val, tol=tol)
    return [(val, vec[:, i == j]) for j, val in enumerate(unique_val)]

---
## 4. `decompose_rep_into_irreps(rep)`

Decompose a reducible representation into its irreducible components. Given 

$\rho = U(\rho_1 \oplus \dots \oplus \rho_r)U^\dagger$, where $\oplus$ = block diagonal construction,  recover 

$\{\rho_1, \dots, \rho_r\}$ up to isomorphism.

$d_i$ is dimension of the irrep $\rho_i$, so $|\rho_i| = d_i$, and $d=|\rho|$. 

$d=\sum_{i=1}^{r}d_i$

**Algorithm sketch:**
1. Find the space of matrices $Q$ satisfying $Q\rho = \rho Q$ using `linalg.infer_change_of_basis(rep, rep)`.
2. Take a random linear combination $\bar{Q} = \sum_i \alpha_i Q_i$ to break accidental degeneracies.
3. Compute the eigenspaces of $\bar{Q}$ — each eigenspace corresponds to an irrep.
4. For each eigenspace with orthonormal basis $B_i$, extract the irrep as $\rho_i = B_i^\dagger \rho B_i$.

Functions you may need: `linalg.infer_change_of_basis`, `np.random.rand`, `np.linalg.eig` or `np.linalg.eigh`, `linalg.eigenspaces`, and `linalg.gram_schmidt`.

<details>
<summary>Code for <code>linalg.eigenspaces</code> and <code>linalg.unique_with_tol</code></summary>

```python
def unique_with_tol(a, *, tol):
    """Find unique elements of an array with a tolerance."""
    assert a.ndim >= 1
    shape = a.shape
    a = a.reshape(len(a), -1)
    distances = np.linalg.norm(a[:, None] - a[None, :], axis=-1)
    inverses = -1 * np.ones(len(a), dtype=int)
    index = 0
    while True:
        (m,) = np.nonzero(inverses == -1)
        if len(m) == 0:
            break
        i = m[0]
        if np.any(inverses[distances[i] < tol] != -1):
            raise ValueError("The clusters are not clearly distinct.")
        inverses[distances[i] < tol] = index
        index += 1
    centers = np.zeros((np.max(inverses) + 1, a.shape[1]), dtype=a.dtype)
    np.add.at(centers, inverses, a)
    centers /= np.bincount(inverses)[:, None]
    centers = centers.reshape(len(centers), *shape[1:])
    return centers, inverses

def eigenspaces(val, vec, *, tol=1e-8):
    """Regroup eigenvectors by eigenvalues."""
    unique_val, i = unique_with_tol(val, tol=tol)
    return [(v, vec[:, i == j]) for j, v in enumerate(unique_val)]
```
</details>

In [ ]:
def decompose_rep_into_irreps(rep: np.array, *, tol: float=1e-08) -> List[np.array]:
    """Decomposes representation into irreducible representations.
    Input:
        rep: np.array [n, d, d] representation of group. rep[g] is a matrix that
            represents g-th element of group.
    Output:
        Irreducible representations. List of np.array [n, d_i, d_i] where d_i is a dimension of i-th irrep.
            Note: you can output the irreps in any order and in any basis.
            If an irrep is included multiple times in the decomposition (i.e. with multiplicity greater than 1), please simply include it multiple times in the output list.
    """
    n = len(rep)
    q = linalg.infer_change_of_basis(rep, rep)
    e_spaces = []
    for i in random.shuffle(list(range(0,n))):
        e_spaces = linalg.eigenspaces()


    pass  # your code here

In [ ]:
# No small tests for this function

---
## 5. `infer_irreps(table)`

Generate the unique irreps of a group from its multiplication table. Start by computing the regular representation, decompose it, then remove duplicates using `are_isomorphic`.

In [ ]:
def infer_irreps(table: np.array, *, tol: float=1e-08) -> List[np.array]:
    """Infers irreducible representations of group represented by multiplication table.
    Input:
        table: np.array [n, n] where table[i, j] = k means i * j = k.
    Output:
        Irreducible representations. List of np.array [n, d, d] where d is a dimension of irrep.
            Note: you can output the irreps in any order and in any basis.
    """
    pass  # your code here

In [ ]:
# No small tests for this functio

---
## 6. `tensor_product(rep1, rep2)`

Compute the tensor product of two representations: $\rho(g) = \rho_1(g) \otimes \rho_2(g)$.

You can either use `np.einsum` followed by a `reshape`, or use `np.kron`.

In [ ]:
def tensor_product(rep1: np.array, rep2: np.array) -> np.array:
    """Returns tensor product of two representations.
    Input:
        rep1: np.array [n, d1, d1] a representation of the group.
        rep2: np.array [n, d2, d2] another representation of the group.
    Output:
        Tensor product of rep1 and rep2. np.array [n, d1*d2, d1*d2], equal to the Kronecker product of rep1[i,:,:] and rep2[i,:,:] at group element i
    """
    pass  # your code here

In [ ]:
np.testing.assert_allclose(
    tensor_product(np.array([[[1.0]], [[-1.0]]]), np.array([[[1.0]], [[-1.0]]])),
    np.array([[[1.0]], [[1.0]]]),
)

---
## 7. `reduce_tensor_product(rep1, rep2, rep3)`

Compute a basis for all similarity transforms between the tensor product of `rep1` and `rep2`, and `rep3`. The output $Q$ satisfies $(\rho_1 \otimes \rho_2) Q = Q \rho_3$.

Clebsch–Gordan coefficients are a special case where `rep3` ranges over all irreps.

You may find `linalg.infer_change_of_basis` useful.

In [ ]:
def reduce_tensor_product(rep1: np.array, rep2: np.array, rep3: np.array) -> np.ndarray:
    """Returns the change of basis matrix that reduces the tensor product of rep1 and rep2 into rep3.
    Input:
        rep1: np.array [n, d1, d1]
        rep2: np.array [n, d2, d2]
        rep3: np.array [n, d3, d3]
    Output:
        Basis of the space of change of basis. mat = np.array [n_sol, d1, d2, d3]
            where each Q=mat[i].reshape(d1*d2,d3) satisfies tensor_product(rep1, rep2) @ Q = Q @ rep3,
            and such that the mat[i] matrices together form a basis for the subspace of all such Q.
    """
    pass  # your code here

In [ ]:
p3_irrep_sign = np.array([[[1.0]], [[-1.0]], [[-1.0]], [[-1.0]], [[1.0]], [[1.0]]])
p3_irrep_rot = np.array([
    [[1.0, 0.0], [0.0, 1.0]],
    [[-0.2916587, 0.95652245], [0.95652245, 0.2916587]],
    [[-0.6825434, -0.73084507], [-0.73084507, 0.6825434]],
    [[0.97420209, -0.22567739], [-0.22567739, -0.97420209]],
    [[-0.5, 0.8660254], [-0.8660254, -0.5]],
    [[-0.5, -0.8660254], [0.8660254, -0.5]],
])
c = reduce_tensor_product(p3_irrep_rot, p3_irrep_sign, p3_irrep_rot)
np.testing.assert_allclose(
    np.einsum('gab,gcd,sace->sgebd', p3_irrep_rot, p3_irrep_sign, c),
    np.einsum('sabc,gdc->sgdab', c, p3_irrep_rot),
)